# 06 — One Copy, Many Engines (capstone)

## Story beat: "The whole platform on one set of files"

This notebook closes Module 1 by showing the **core Fabric promise**: the Delta files we wrote in OneLake are consumed by **Spark** (here), **T-SQL** (lakehouse SQL endpoint + warehouse in Module 2), **Power BI Direct Lake** (Module 4), and — after mirroring — **operational SQL data** blended with analytics (Module 3).

No ETL copies. No "sync to the reporting database." One truth.

---

## The engines that share this data

| Engine | How it reads gold | Module |
| --- | --- | --- |
| **Spark** | `spark.table("gold.sales_by_store_day")` | This notebook |
| **Lakehouse SQL** | `SELECT * FROM gold.sales_by_store_day` (read-only) | Module 2 |
| **Warehouse T-SQL** | `SELECT * FROM lh_retail.gold.sales_by_region` (cross-item) | Module 2 |
| **Direct Lake Power BI** | Semantic model storage mode = Direct Lake | Module 4 |
| **Mirrored OLTP** | Shortcut to `sqldb_orders` Delta | Module 3 + optional cell below |

In [ ]:
%run 00_setup

## Step 1 — Spark reads gold (analytics path)

Aggregate net sales by **region** — the same question Module 4 will answer in a report visual.

In [ ]:
gold = spark.table(f"{GOLD_SCHEMA}.sales_by_store_day")
print("rows:", gold.count())
display(
    gold.groupBy("region").sum("net_sales").orderBy("sum(net_sales)", ascending=False)
)

## Step 2 — (Optional) Blend mirrored operational orders

After **Module 3**, `sqldb_orders` mirrors `dbo.Orders` to Delta in OneLake (~30s). Create a **shortcut** in `lh_retail` → `Files/orders_shortcut` pointing at that mirror, then run this cell.

This is **translytical** analytics: POS batch data (gold) + live app orders (mirrored OLTP) in one Spark session.

In [ ]:
try:
    orders = spark.read.format("delta").load("Files/orders_shortcut/Orders")
    display(orders.groupBy("store_id").sum("order_total"))
except Exception as e:
    print("Mirrored orders not shortcutted yet — complete Module 3 first, or skip.")
    print(e)

## Step 3 — Publish `gold.sales_by_region` for the warehouse

We write one more curated table. In **Module 2** you'll query it from `wh_retail` with three-part naming:

```sql
SELECT * FROM lh_retail.gold.sales_by_region;
```

Same Delta files — warehouse engine, zero copy.

In [ ]:
from pyspark.sql import functions as F

curated = (
    gold.groupBy("region")
    .agg(
        F.sum("net_sales").alias("net_sales"),
        F.sum("transactions").alias("transactions"),
    )
)
curated.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_SCHEMA}.sales_by_region")
print("Wrote gold.sales_by_region — query from wh_retail in Module 2.")

---

## ✅ Module 1 complete — what you built

| Layer | Tables | Purpose |
| --- | --- | --- |
| Bronze | stores, products, sales | Raw POS landing |
| Silver | dim_store, dim_product, fact_sales | Clean dimensional model + V-Order |
| Gold | sales_by_store_day, sales_by_category, sales_by_region | BI-ready data products |

**Continue to Module 2** — open `wh_retail` and join warehouse tables to these lakehouse tables in pure T-SQL.